In [27]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import tqdm

In [4]:
data = pd.read_parquet('../data/features/feature_matrix.parquet')

In [5]:
scaler = StandardScaler()
data = scaler.fit_transform(data)

In [7]:
class Model(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 8)
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, input_dim)
        )
    
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon

In [8]:
model = Model(29)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_func = nn.MSELoss()

In [24]:
X_tensor = torch.FloatTensor(data)

dataset = TensorDataset(X_tensor)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=5)

In [25]:
model.train()

for epoch in range(50):
    lm_count = 0
    loss_mean = 0
    vis_data = tqdm.tqdm(loader, leave=True)

    for (x_train, ) in vis_data:
        x_recon = model(x_train)
        loss = loss_func(x_recon, x_train)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (lm_count - 1)/lm_count * loss_mean
        vis_data.set_description(f'loss_mean: {loss_mean:.4f}')

    scheduler.step(loss_mean)



loss_mean: 0.0236: 100%|██████████| 1438/1438 [00:01<00:00, 832.43it/s]


In [26]:
model.eval()
with torch.no_grad():
    X_recon = model(X_tensor)
    ae_errors = torch.mean((X_tensor - X_recon) ** 2, dim=1).numpy()

print(f"ae_score: min={ae_errors.min():.4f}, max={ae_errors.max():.4f}, mean={ae_errors.mean():.4f}")

ae_score: min=0.0005, max=28.2534, mean=0.0236


In [29]:
minscale = MinMaxScaler(feature_range=(0,100))

data_full = pd.read_parquet("../data/features/data_with_scores.parquet")
data_full["ae_score"] = minscale.fit_transform(ae_errors.reshape(-1, 1)).flatten()


In [30]:
top35_ae = data_full.nlargest(35, "ae_score")[[
    "period", "account_dt", "account_kt", "amount",
    "contractor", "ТипДокумента", "ae_score", "ensemble_score"
]]
print(top35_ae.to_string())

                    period account_dt account_kt      amount                                                contractor                       ТипДокумента    ae_score  ensemble_score
38539  2024-03-31 00:00:00         94      41.01   -90556.36                                       Внутренняя операция                           Операция  100.000000        0.500000
332834 2025-10-21 23:59:59      10.01      60.01    10500.00                                 ПЕТРОВА ЮЛИЯ СЕРГЕЕВНА ИП  Поступление (акт, накладная, УПД)   65.512756        0.472856
155396 2024-11-12 23:59:59         51      62.01   117735.00                                 Слепцова Ольга Николаевна      Поступление на расчетный счет   63.949734        0.356579
238224 2025-04-30 15:00:00      91.02      76.02     -160.68                                       Внутренняя операция                           Операция   63.799660        0.358371
1136   2024-01-09 23:59:59      57.01      50.01   420000.00                              